# 1. Exploratory Data Analysis (EDA)

## 1.1 Import Libraries and Load Data
First, I'll import the necessary libraries and load my dataset from the `data.csv` file.

In [ ]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset from the CSV file
df = pd.read_csv('data.csv')

# Show the first 5 rows to check if it loaded
df.head()

## 1.2 Check Data Structure & Missing Values

Here, I'll run `df.info()` to see the data types of each column (like text or number) and to find out how many values are missing.

In [ ]:
# Get a summary of all columns (types, missing values)
df.info()

## 1.3 Check My Target Variable (Loan_Status)

Now I'll count the 'Y' (Yes) and 'N' (No) values in my target column (`Loan_Status`) to see if the dataset is imbalanced.

In [ ]:
# Count the 'Y' and 'N' values in our target column
print(df['Loan_Status'].value_counts())

# Plot the target variable
sns.countplot(x='Loan_Status', data=df)
plt.show()

## 1.4 Check Numeric Columns

This gives me a quick statistical summary (like mean, min, max) for all columns that contain numbers.

In [ ]:
# Get quick stats (mean, min, max) for number columns
df.describe()

## 1.5 Check Categorical (Text) Columns

Finally, I'll look at the different categories in my main text-based columns.

In [ ]:
# Check unique values in a few text columns
print("--- Gender Counts ---")
print(df['Gender'].value_counts())

print("\n--- Education Counts ---")
print(df['Education'].value_counts())

print("\n--- Property_Area Counts ---")
print(df['Property_Area'].value_counts())

# 2. Data Cleaning & Pre-processing

Now that I've explored the data, I need to clean it.
My plan is to:
1.  Fill in all the missing values (imputation).
2.  Convert all text columns (categorical) into numbers (encoding).

## 2.1 Handle Missing Values

Based on my `df.info()` output, I have missing values in several columns. I'll fill them in using the most logical value.
* For number columns like `LoanAmount`, I'll use the **median** (the middle value) because the `df.describe()` output showed it has outliers.
* For text columns like `Gender` or `Married`, I'll use the **mode** (the most common value).

In [ ]:
# --- Fill Missing Numeric Values ---
# Fill missing 'LoanAmount' with the median (middle) value
df['LoanAmount'] = df['LoanAmount'].fillna(df['LoanAmount'].median())

# Fill missing 'Loan_Amount_Term' with the mode (most common) value
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].mode()[0])

# Fill missing 'Credit_History' with the mode (most common) value (which is 1.0)
df['Credit_History'] = df['Credit_History'].fillna(df['Credit_History'].mode()[0])

# --- Fill Missing Categorical (Text) Values ---
# Fill 'Gender', 'Married', 'Dependents', and 'Self_Employed' with the mode
df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])
df['Married'] = df['Married'].fillna(df['Married'].mode()[0])
df['Dependents'] = df['Dependents'].fillna(df['Dependents'].mode()[0])
df['Self_Employed'] = df['Self_Employed'].fillna(df['Self_Employed'].mode()[0])

# --- Check if all missing values are fixed ---
print("--- Missing Values After Cleaning ---")
print(df.isnull().sum())

## 2.2 Convert Categorical (Text) Data to Numbers

My model can't understand text like 'Male' or 'Y'. I need to convert these into numbers.
* `Loan_Status` (my target) will become: `Y`=1, `N`=0.
* Other columns like `Gender` will become: `Male`=1, `Female`=0.
* `Dependents` and `Property_Area` will also be converted.

In [ ]:
# --- Convert Target Variable ---
# Change 'Y' and 'N' in 'Loan_Status' to 1 and 0
df['Loan_Status'] = df['Loan_Status'].map({'Y': 1, 'N': 0})

# --- Convert Other Categorical Columns ---
# Change 'Gender' to 1 (Male) and 0 (Female)
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})

# Change 'Married' to 1 (Yes) and 0 (No)
df['Married'] = df['Married'].map({'Yes': 1, 'No': 0})

# Change 'Education' to 1 (Graduate) and 0 (Not Graduate)
df['Education'] = df['Education'].map({'Graduate': 1, 'Not Graduate': 0})

# Change 'Self_Employed' to 1 (Yes) and 0 (No)
df['Self_Employed'] = df['Self_Employed'].map({'Yes': 1, 'No': 0})

# 'Dependents' has '3+' which isn't a number. We'll replace '3+' with 3.
df['Dependents'] = df['Dependents'].replace('3+', '3')
# Now convert the whole column to a number
df['Dependents'] = pd.to_numeric(df['Dependents'])

# 'Property_Area' has 3 values. We'll use get_dummies to convert it
# This creates 3 new columns (Urban, Rural, Semiurban) with 1s and 0s
df = pd.get_dummies(df, columns=['Property_Area'])

# --- Check our work ---
print("\n--- Data Head After Encoding ---")
df.head()

## 2.3 Final Check

Now I'll run `df.info()` one last time to confirm all data is numeric and there are no missing values.

In [ ]:
# --- Final Check ---
# All columns should now have 614 non-null values
# All columns should be a number (int64, float64, or uint8)
df.info()

# 3. Model Building & Training

Now that my data is 100% clean and numeric, I can start building the machine learning models.

## 3.1 Define Features (X) and Target (y)

First, I need to separate my data into two new tables:
1.  `X`: The **features** (all the columns I'll use to make the prediction, like `ApplicantIncome`, `Credit_History`, `Gender`, etc.).
2.  `y`: The **target** (the single "answer" column I want to predict, which is `Loan_Status`).

In [ ]:
# --- 1. Define Target (y) ---
# 'y' is just the 'Loan_Status' column
y = df['Loan_Status']

# --- 2. Define Features (X) ---
# 'X' is all the other columns, except 'Loan_ID' which has no predictive value.
X = df.drop(['Loan_Status', 'Loan_ID'], axis=1)

# --- 3. Check our work ---
print("--- Shape of X (Features) ---")
print(X.shape)
print("\n--- Shape of y (Target) ---")
print(y.shape)

## 3.2 Split Data into Train/Test Sets

I need to split my data into two parts so I can test my model fairly.
1.  **Training Set (80%):** The model will "learn" from this data.
2.  **Testing Set (20%):** The model will be tested on this "unseen" data to see how accurate it is.

In [ ]:
# Import the train_test_split function from scikit-learn
from sklearn.model_selection import train_test_split

# Split the data: 80% for training, 20% for testing
# 'random_state=42' ensures I get the same "random" split every time I run this
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Check the shapes to confirm
print("X_train (features) shape:", X_train.shape)
print("X_test (features) shape:", X_test.shape)
print("y_train (target) shape:", y_train.shape)
print("y_test (target) shape:", y_test.shape)

## 3.3 Scale the Data

My `df.describe()` showed that columns like `ApplicantIncome` (150-81000) and `Credit_History` (0-1) are on very different scales. This can confuse the model.

I will use `StandardScaler` to scale all my features to a similar range. This will fix the `ConvergenceWarning` and often improves model accuracy.

In [ ]:
# --- 1. Import the Scaler ---
from sklearn.preprocessing import StandardScaler

# --- 2. Create and Fit the Scaler ---
# Create an instance of the scaler
scaler = StandardScaler()

# Fit the scaler ONLY on the training data (to learn the mean and standard deviation)
scaler.fit(X_train)

# --- 3. Apply the Scaler ---
# Transform both the training and testing data
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data successfully scaled!")

## 3.4 Model 1: Logistic Regression

Now I'll train my first model. I'm starting with Logistic Regression because it's effective and highly **interpretable** (I can see *why* it makes its decisions).

In [ ]:
# --- 1. Import the Model ---
from sklearn.linear_model import LogisticRegression

# --- 2. Create the Model ---
# We can keep max_iter=1000 just to be safe, but the scaling should fix the problem
model_logreg = LogisticRegression(max_iter=1000)

# --- 3. Train the Model ---
# **CRITICAL:** Fit the model on the SCALED training data (X_train_scaled)
model_logreg.fit(X_train_scaled, y_train)

print("Logistic Regression Model Trained Successfully on Scaled Data!")